# 📘 색상 검출과 도형 그리기

특정 색상을 검출하거나 이미지에 도형을 그리는 실전 기법을 배웁니다.

**학습 목표:**
- HSV 색상 공간을 활용한 색상 검출
- 마스크와 비트 연산
- 도형 그리기 (선, 사각형, 원, 텍스트)
- 트랙바를 활용한 실시간 매개변수 조절

## 1. HSV 색상 공간으로 색상 검출

HSV(Hue-Saturation-Value)는 색상을 직관적으로 표현합니다:
- **H**(Hue): 색상 (0~179 in OpenCV)
- **S**(Saturation): 채도 (0~255)
- **V**(Value): 명도 (0~255)

> 💡 BGR에서 특정 색상을 검출하기 어려운 이유는 빛의 밝기에 따라
> R, G, B 값이 모두 변하기 때문입니다. HSV에서는 H 값만으로 색상을 분리할 수 있습니다.

In [ ]:
# ┌─────────────────────────────────────────┐
# │  HSV 색상 검출                           │
# │  특정 색상(빨강, 파랑, 초록)만 추출       │
# └─────────────────────────────────────────┘

# 컬러풀한 테스트 이미지 생성
img = np.zeros((300, 400, 3), dtype=np.uint8)
img[:] = (200, 200, 200)  # 밝은 회색 배경
cv2.rectangle(img, (20, 20), (120, 280), (0, 0, 255), -1)   # 빨강 (BGR)
cv2.circle(img, (200, 150), 80, (255, 0, 0), -1)              # 파랑
cv2.rectangle(img, (300, 50), (380, 250), (0, 255, 0), -1)   # 초록

# BGR → HSV 변환
hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

# 빨강 검출 (H: 0~10, 170~180)
mask_red1 = cv2.inRange(hsv, np.array([0, 100, 100]), np.array([10, 255, 255]))
mask_red2 = cv2.inRange(hsv, np.array([170, 100, 100]), np.array([180, 255, 255]))
mask_red = mask_red1 | mask_red2

# 파랑 검출 (H: 100~130)
mask_blue = cv2.inRange(hsv, np.array([100, 100, 100]), np.array([130, 255, 255]))

# 초록 검출 (H: 35~85)
mask_green = cv2.inRange(hsv, np.array([35, 100, 100]), np.array([85, 255, 255]))

# 마스크 적용 (비트 연산)
result_red = cv2.bitwise_and(img, img, mask=mask_red)
result_blue = cv2.bitwise_and(img, img, mask=mask_blue)
result_green = cv2.bitwise_and(img, img, mask=mask_green)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax in axes.flat:
    ax.axis('off')
axes[0, 0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); axes[0, 0].set_title('원본')
axes[0, 1].imshow(mask_red, cmap='gray'); axes[0, 1].set_title('빨강 마스크')
axes[0, 2].imshow(cv2.cvtColor(result_red, cv2.COLOR_BGR2RGB)); axes[0, 2].set_title('빨강 검출')
axes[1, 0].imshow(mask_blue, cmap='gray'); axes[1, 0].set_title('파랑 마스크')
axes[1, 1].imshow(cv2.cvtColor(result_blue, cv2.COLOR_BGR2RGB)); axes[1, 1].set_title('파랑 검출')
axes[1, 2].imshow(cv2.cvtColor(result_green, cv2.COLOR_BGR2RGB)); axes[1, 2].set_title('초록 검출')
plt.tight_layout()
plt.show()

## 2. 도형 그리기와 텍스트

OpenCV로 이미지 위에 도형과 텍스트를 그릴 수 있습니다.
이는 결과 시각화, ROI 표시 등에 유용합니다.

In [ ]:
# ┌─────────────────────────────────────────┐
# │  도형 그리기와 텍스트                     │
# │  선, 사각형, 원, 다각형, 텍스트          │
# └─────────────────────────────────────────┘

# 400×600 흰 배경
canvas = np.ones((400, 600, 3), dtype=np.uint8) * 255

# 선 그리기
cv2.line(canvas, (50, 50), (250, 50), (255, 0, 0), 3)          # 파란 선
cv2.line(canvas, (50, 80), (250, 80), (0, 200, 0), 2, cv2.LINE_AA)  # 초록 선(안티앨리어싱)

# 사각형
cv2.rectangle(canvas, (50, 120), (200, 220), (0, 0, 255), 3)   # 빨간 테두리
cv2.rectangle(canvas, (220, 120), (370, 220), (0, 165, 255), -1)  # 주황 채우기

# 원
cv2.circle(canvas, (130, 300), 60, (255, 0, 255), 3)            # 보라 테두리
cv2.circle(canvas, (310, 300), 60, (0, 255, 255), -1)          # 노란 채우기

# 타원
cv2.ellipse(canvas, (490, 170), (80, 40), 30, 0, 360, (128, 0, 128), 2)

# 다각형
pts = np.array([[450, 280], [550, 280], [580, 380], [420, 380]], np.int32)
cv2.polylines(canvas, [pts], True, (0, 128, 255), 3)

# 텍스트
cv2.putText(canvas, 'OpenCV Drawing', (50, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 0), 2)
cv2.putText(canvas, 'Korean? X', (300, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (100, 100, 100), 1)

fig, ax = plt.subplots(figsize=(10, 7))
ax.imshow(cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB))
ax.set_title('OpenCV 도형 그리기')
ax.axis('off')
plt.tight_layout()
plt.show()

print("💡 cv2.line(): 선, cv2.rectangle(): 사각형, cv2.circle(): 원")
print("💡 두께 매개변수: -1 = 채우기, 양수 = 테두리 두께")
print("💡 cv2.LINE_AA: 안티앨리어싱(부드러운 선)")

## 3. 윤곽선 검출

윤곽선(contour)은 객체의 외곽선을 검출합니다.
객체의 면적, 중심, 둘레 등을 계산할 수 있습니다.

In [ ]:
# ┌─────────────────────────────────────────┐
# │  윤곽선 검출과 분석                      │
# │  findContours, drawContours, 영역 계산    │
# └─────────────────────────────────────────┘

# 여러 도형 이미지 생성
img = np.zeros((300, 400, 3), dtype=np.uint8)
cv2.rectangle(img, (30, 30), (130, 130), (255, 255, 255), -1)
cv2.circle(img, (250, 100), 60, (255, 255, 255), -1)
cv2.rectangle(img, (300, 200), (380, 280), (255, 255, 255), -1)
cv2.circle(img, (80, 230), 40, (255, 255, 255), -1)

# 흑백 변환 후 윤곽선 검출
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
contours, hierarchy = cv2.findContours(gray, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

# 윤곽선 분석
result = img.copy()
print(f"검출된 윤곽선 수: {len(contours)}")
print(f"{'번호':>4s} {'면적':>8s} {'둘레':>8s} {'중심':>12s}")
print("-" * 40)

for i, cnt in enumerate(contours):
    area = cv2.contourArea(cnt)
    perimeter = cv2.arcLength(cnt, True)
    M = cv2.moments(cnt)
    if M['m00'] > 0:
        cx = int(M['m10'] / M['m00'])
        cy = int(M['m01'] / M['m00'])
    else:
        cx, cy = 0, 0
    print(f"{i:>4d} {area:>8.0f} {perimeter:>8.1f} ({cx:>4d}, {cy:>4d})")
    
    # 윤곽선 그리기
    cv2.drawContours(result, [cnt], -1, (0, 255, 0), 2)
    # 중심 표시
    cv2.circle(result, (cx, cy), 3, (0, 0, 255), -1)

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(gray, cmap='gray'); axes[0].set_title('이진화 이미지')
axes[1].imshow(cv2.cvtColor(result, cv2.COLOR_BGR2RGB)); axes[1].set_title('윤곽선 + 중심점')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

## 🎯 연습 문제

1. 빨강, 초록, 파랑 세 개의 원이 있는 이미지에서 각 색상을 개별적으로 검출하세요.
2. 검출된 객체의 바운딩 박스(외접 사각형)를 그리세요 (`cv2.boundingRect()`).
3. 이미지에 한국어 텍스트를 그리는 방법을 조사해보세요 (PIL 사용).
4. 웹캠이나 이미지에서 피부색(Hue 0~20)을 검출하는 마스크를 만드세요.
5. 윤곽선 검출로 가장 큰 면적의 객체만 찾아 강조 표시하세요.